# Cynbit DocuMind AI\n## Notebook 01: Document Ingestion\n\n### Objective\nThis notebook builds the first stage of DocuMind: converting a PDF into page-level, machine-readable records. Each record retains text, source filename, page number, character count, document metadata, and an empty-page flag.\n\n### Why this matters\nRetrieval and answer generation need traceable source text. Keeping page boundaries now lets later RAG answers cite the page that supported them.

### 1. Imports\nPyMuPDF reads PDF pages; `Path` keeps paths portable; pandas provides a compact inspection table.\n\n**Expected output:** `All libraries imported successfully!`

In [ ]:
import sys\nfrom pathlib import Path\n\nimport pandas as pd\n\nPROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()\nif str(PROJECT_ROOT) not in sys.path:\n    sys.path.insert(0, str(PROJECT_ROOT))\n\nfrom app.ingestion.pdf_loader import extract_pdf_text\nprint('All libraries imported successfully!')

### 2. Configure project paths\nUsing paths relative to the project root avoids hard-coding a specific Mac username or folder.

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data'\nSAMPLE_DOCS_DIR = DATA_DIR / 'sample_documents'\nSAMPLE_DOCS_DIR.mkdir(parents=True, exist_ok=True)\nprint('Project Root:', PROJECT_ROOT)\nprint('Sample Documents:', SAMPLE_DOCS_DIR)

### 3. Extract and inspect a PDF\n`extract_pdf_text` returns a list rather than one large string, so every text segment stays linked to its originating page. Blank or scanned pages are safely retained with `has_text=False`; OCR can be added later.\n\n**Validation:** the summary should show one row per PDF page, and page numbers should start at 1.

In [ ]:
def inspect_extraction(extracted_pages):\n    return pd.DataFrame([\n        {key: page[key] for key in ('page_number', 'character_count', 'has_text', 'source_file')}\n        for page in extracted_pages\n    ])\n\npdf_files = sorted(SAMPLE_DOCS_DIR.glob('*.pdf'))\nprint('PDF documents found:', len(pdf_files))\nif pdf_files:\n    extracted_pages = extract_pdf_text(pdf_files[0])\n    display(inspect_extraction(extracted_pages))\n    print('\\nFirst page preview:\n')\n    print(extracted_pages[0]['text'][:1000])\nelse:\n    print('Add a PDF to data/sample_documents and run this cell again.')

### 4. Automated validation\nRun the following from the project root to validate normal extraction, blank-page handling, missing files, and non-PDF rejection: `venv/bin/python -m pytest -q`.

## 9. Text Preprocessing\nPDF text can contain inconsistent Unicode, excess whitespace, artificial line breaks, and words split by hyphenation. We normalize it before embedding, while retaining paragraph boundaries because they often carry meaning.\n\n**Why:** clean, consistent text improves retrieval quality without losing source-page metadata.

In [ ]:
from app.processing.text_processing import (\n    clean_pages, clean_text, create_document_chunks, save_chunks,\n    split_text_into_chunks, validate_chunks,\n)\n\nsample_text = 'PROJECT REQUIREMENTS\n\nThe   client requires require-\nments from different departments.'\nprint(clean_text(sample_text))

**Expected output:** `requirements` appears as one word, repeated spaces are removed, and the blank line remains a paragraph break. The cleaner normalizes Unicode, fixes a hyphen immediately followed by a line break, and joins wrapped lines.

## 10. Apply preprocessing while preserving metadata\nCleaning must not sever the relationship between text and its PDF page. `clean_pages` copies every page record, changing only the text-derived fields.

In [ ]:
cleaned_pages = clean_pages(extracted_pages)\ncomparison = pd.DataFrame([\n    {'page_number': raw['page_number'], 'raw_characters': len(raw['text']), 'cleaned_characters': len(clean['text'])}\n    for raw, clean in zip(extracted_pages, cleaned_pages)\n])\ndisplay(comparison)\nassert [page['page_number'] for page in cleaned_pages] == [page['page_number'] for page in extracted_pages]

## 11. Boundary-aware chunking\nLong documents are too large to pass to a model on every question. We use approximate 1,000-character chunks with 150 shared characters. The splitter prefers paragraphs, then lines, sentences, and finally whitespace, preserving context across chunks.

In [ ]:
CHUNK_SIZE = 1000\nCHUNK_OVERLAP = 150\nassert CHUNK_SIZE > 0 and 0 <= CHUNK_OVERLAP < CHUNK_SIZE\nsample_chunks = split_text_into_chunks(' '.join(['A sentence about attendance management.'] * 20), 150, 30)\nprint('Chunks:', len(sample_chunks))\nprint('First chunk length:', len(sample_chunks[0]))\nassert all(chunk.strip() and len(chunk) <= 150 for chunk in sample_chunks)

**Expected output:** multiple non-empty chunks, each no longer than 150 characters in this deliberately small demonstration. The production configuration above uses 1,000/150.

## 12. Chunk metadata, validation, and persistence\nEach chunk receives a unique ID, source document, original page number, position on that page, character count, and document metadata. This is what makes later citations possible.

In [ ]:
source_document = pdf_files[0].name\ndocument_chunks = create_document_chunks(cleaned_pages, source_document)\nvalidate_chunks(document_chunks)\nchunks_df = pd.DataFrame(document_chunks)\ndisplay(chunks_df[['chunk_id', 'page_number', 'chunk_index', 'character_count']])\n\nOUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'\noutput_path = save_chunks(document_chunks, OUTPUT_DIR / f'{Path(source_document).stem}_chunks.json')\nprint('All chunk validation tests passed!')\nprint('Chunks saved to:', output_path)

### Validation summary\nThe assertions above ensure non-empty text and valid settings. `validate_chunks` additionally checks unique IDs, sources, positive page numbers, and accurate character counts. Run `venv/bin/python -m pytest -q` from the project root for automated edge-case tests.\n\nCurrent limitation: chunking happens page by page, so a sentence spanning pages remains separate. Character counts are also an approximation of model tokens.